# 🌾 Germinated and Non-Germinated Seed Dataset Analysis

This notebook provides exploratory data analysis (EDA) and visualizations for the **Germinated and Non-Germinated Seed Dataset**.

### Objectives:
1. **Summary Statistics**: Inspect image counts and label annotations across dataset splits (`train`, `val`, `test`).
2. **Class Distribution Analysis**: Analyze the balance between **Germinated** (Class 0) and **Non-Germinated** (Class 1) seeds.
3. **Bounding Box Analysis**: Examine bounding box sizes, aspect ratios, and seed density per image.
4. **Visual Overlays**: Render sample images with colored bounding box annotations (Green 🌱 for Germinated, Red 🛑 for Non-Germinated).

## 1. Environment Setup & Dependencies

In [ ]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
print("Libraries loaded successfully!")

## 2. Dataset Path Configuration & Split Inspection

In [ ]:
# Base dataset directory path
DATASET_DIR = os.path.abspath("dataset")
SPLITS = ['train', 'val', 'test']
CLASS_NAMES = {0: 'Germinated', 1: 'Non-Germinated'}

split_summary = []
for split in SPLITS:
    img_dir = os.path.join(DATASET_DIR, split, "image")
    lbl_dir = os.path.join(DATASET_DIR, split, "label")
    
    imgs = glob.glob(os.path.join(img_dir, "*.*"))
    lbls = glob.glob(os.path.join(lbl_dir, "*.txt"))
    
    split_summary.append({
        'Split': split.capitalize(),
        'Images': len(imgs),
        'Label Files': len(lbls)
    })

df_splits = pd.DataFrame(split_summary)
print("=== Dataset Split Summary ===")
display(df_splits)

## 3. Extracting Bounding Box & Class Data

In [ ]:
records = []
img_stats = []

for split in SPLITS:
    lbl_dir = os.path.join(DATASET_DIR, split, "label")
    lbl_files = glob.glob(os.path.join(lbl_dir, "*.txt"))
    
    for lbl_path in lbl_files:
        file_name = os.path.basename(lbl_path)
        count_0, count_1 = 0, 0
        
        with open(lbl_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    xc, yc, w, h = [float(v) for v in parts[1:5]]
                    
                    if cls_id == 0:
                        count_0 += 1
                    elif cls_id == 1:
                        count_1 += 1
                        
                    records.append({
                        'split': split,
                        'file_name': file_name,
                        'class_id': cls_id,
                        'class_name': CLASS_NAMES.get(cls_id, f'Class {cls_id}'),
                        'x_center': xc,
                        'y_center': yc,
                        'width': w,
                        'height': h,
                        'area': w * h,
                        'aspect_ratio': w / h if h > 0 else 0
                    })
                    
        total_seeds = count_0 + count_1
        germ_rate = (count_0 / total_seeds * 100.0) if total_seeds > 0 else 0
        img_stats.append({
            'split': split,
            'file_name': file_name,
            'germinated': count_0,
            'non_germinated': count_1,
            'total_seeds': total_seeds,
            'germination_rate_pct': germ_rate
        })

df_boxes = pd.DataFrame(records)
df_images = pd.DataFrame(img_stats)

print(f"Total Bounding Boxes Extracted: {len(df_boxes)}")
print(f"Total Annotated Images: {len(df_images)}")
display(df_boxes.head(10))

## 4. Class Distribution Visualizations

In [ ]:
# Plot Class Counts (Germinated vs Non-Germinated)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot by split
sns.countplot(data=df_boxes, x='split', hue='class_name', palette={'Germinated': '#2ecc71', 'Non-Germinated': '#e74c3c'}, ax=axes[0])
axes[0].set_title('Seed Bounding Box Counts by Dataset Split', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Dataset Split')
axes[0].set_ylabel('Number of Seeds')

# Overall Pie chart
class_counts = df_boxes['class_name'].value_counts()
axes[1].pie(class_counts, labels=class_counts.index, autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=140, explode=(0.03, 0.03))
axes[1].set_title('Overall Class Distribution', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Seed Density & Bounding Box Size Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of total seeds per image
sns.histplot(df_images['total_seeds'], bins=10, kde=True, color='#3498db', ax=axes[0])
axes[0].set_title('Distribution of Seeds per Image', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Total Seeds in Image')
axes[0].set_ylabel('Image Count')

# Bounding box normalized width vs height scatter plot
sns.scatterplot(data=df_boxes, x='width', y='height', hue='class_name', alpha=0.6, palette={'Germinated': '#2ecc71', 'Non-Germinated': '#e74c3c'}, ax=axes[1])
axes[1].set_title('Bounding Box Width vs. Height (Normalized)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Normalized Box Width')
axes[1].set_ylabel('Normalized Box Height')

plt.tight_layout()
plt.show()

## 6. Visualizing Sample Images with Bounding Box Overlays

In [ ]:
def draw_bounding_boxes(image_path, label_path):
    """Helper function to load image and draw color-coded bounding boxes."""
    img = cv2.imread(image_path)
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h_img, w_img, _ = img.shape
    
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    cls_id = int(parts[0])
                    xc, yc, bw, bh = [float(v) for v in parts[1:5]]
                    
                    x1 = int((xc - bw / 2) * w_img)
                    y1 = int((yc - bh / 2) * h_img)
                    x2 = int((xc + bw / 2) * w_img)
                    y2 = int((yc + bh / 2) * h_img)
                    
                    # Color: Green for Germinated (0), Red for Non-Germinated (1)
                    color = (46, 204, 113) if cls_id == 0 else (231, 76, 60)
                    label_str = "Germinated" if cls_id == 0 else "Non-Germinated"
                    
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                    cv2.putText(img, label_str, (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
                    
    return img

# Display 4 sample images from train set
sample_files = glob.glob(os.path.join(DATASET_DIR, "train", "image", "*.*"))[:4]

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
for idx, img_path in enumerate(sample_files):
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = os.path.join(DATASET_DIR, "train", "label", f"{base_name}.txt")
    
    annotated_img = draw_bounding_boxes(img_path, lbl_path)
    ax = axes[idx // 2, idx % 2]
    if annotated_img is not None:
        ax.imshow(annotated_img)
        ax.set_title(f"Sample: {base_name}", fontsize=12, fontweight='bold')
    ax.axis('off')

plt.suptitle("Sample Dataset Images with Bounding Box Annotations\n(Green = Germinated 🌱 | Red = Non-Germinated 🛑)", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()